# UniAssist — Week 1: Corpus and Textual Sources
The corpus contains 20 English source-based texts about University Metropolitan Tirana (UMT).
The source metadata records URLs and flags time-sensitive policies.
Complete the discussion questions after running the cells; generated counts are not manual gold judgments.


In [1]:
from pathlib import Path
import json, re, statistics
from collections import Counter
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "corpus").exists():
    ROOT = ROOT.parent
assert (ROOT / "corpus").exists(), "Run notebook from UniAssist_UMT directory"
metadata = json.loads((ROOT / "metadata.json").read_text(encoding="utf-8"))
records = metadata["documents"]
texts = [(ROOT / "corpus" / row["file"]).read_text(encoding="utf-8") for row in records]
print(len(texts), "documents loaded")


20 documents loaded


## Task 1 — Inventory, word counts, source genres and quality


In [2]:
rows = []
for entry, text in zip(records, texts):
    rows.append({"filename": entry["file"], "genre": entry["genre"],
                 "chars": len(text), "whitespace_tokens": len(text.split()),
                 "source_count": len(entry["sources"]),
                 "time_sensitive": entry["time_sensitive"]})
table = pd.DataFrame(rows)
display(table)
print("Total words:", table["whitespace_tokens"].sum())
print("Genres:", table["genre"].value_counts().to_dict())
print("Exact duplicate texts:", len(texts)-len(set(texts)))


,filename,genre,chars,whitespace_tokens,source_count,time_sensitive
0,01_university_overview.txt,information_article,1968,251,2,False
1,02_computer_science_faculty.txt,information_article,2074,266,1,True
2,03_engineering_architecture_faculty.txt,information_article,1956,249,1,True
3,04_economics_faculty.txt,information_article,1924,254,1,True
4,05_english_taught_programs.txt,information_article,1903,257,1,True
5,06_bachelor_admissions_faq.txt,visitor_faq,1757,248,2,True
6,07_bachelor_documents_faq.txt,visitor_faq,1756,244,2,True
7,08_master_admissions_faq.txt,visitor_faq,1823,247,1,True
8,09_professional_doctoral_studies.txt,information_article,1823,250,2,True
9,10_scholarships_faq.txt,visitor_faq,1800,244,1,True


Total words: 5068
Genres: {'information_article': 9, 'visitor_faq': 6, 'policy_summary': 5}
Exact duplicate texts: 0


## Task 2 — Tokens, types and common words using spaCy


In [3]:
import spacy
nlp = spacy.blank("en")
all_tokens = [tok.text.lower() for text in texts for tok in nlp.make_doc(text) if not tok.is_space]
alpha = [tok for tok in all_tokens if tok.isalpha()]
print("Alphabetic tokens:", len(alpha), "types:", len(set(alpha)))
print("Type/token ratio:", round(len(set(alpha)) / max(1, len(alpha)), 4))
print("Top 15:", Counter(alpha).most_common(15))
filtered = [tok for tok in alpha if tok not in nlp.Defaults.stop_words]
print("Top 15 without stopwords:", Counter(filtered).most_common(15))


Alphabetic tokens: 5098 types: 1127
Type/token ratio: 0.2211
Top 15: [('the', 254), ('and', 210), ('a', 187), ('of', 75), ('for', 71), ('to', 62), ('page', 55), ('should', 52), ('or', 51), ('an', 48), ('in', 47), ('university', 46), ('not', 45), ('programme', 45), ('is', 43)]
Top 15 without stopwords: [('page', 55), ('university', 46), ('programme', 45), ('umt', 37), ('official', 37), ('academic', 37), ('student', 37), ('published', 31), ('study', 28), ('master', 28), ('year', 27), ('programmes', 26), ('professional', 24), ('faculty', 24), ('engineering', 23)]


## Task 3 — UTF-8, punctuation and difficult text


In [4]:
print("All files decoded as UTF-8")
for row, text in zip(records, texts):
    if re.search(r"[À-ÿ–—+]", text):
        print(row["file"], "has accents or special symbols")
print("Potential replacement characters:", sum(text.count("�") for text in texts))


All files decoded as UTF-8
11_study_fees_student_access_faq.txt has accents or special symbols
12_academic_calendar_2026_2027.txt has accents or special symbols
14_examination_regulation_scope.txt has accents or special symbols
19_campus_laboratories_library.txt has accents or special symbols
20_erasmus_mobility_information.txt has accents or special symbols
Potential replacement characters: 0


## Task 4 — Dependency parsing and named entities
Install the model with: python -m spacy download en_core_web_sm.
Mark at least one correct and one incorrect result for each of three sentences.


In [5]:
from spacy import displacy
try:
    trained = spacy.load("en_core_web_sm")
except OSError:
    print("Install en_core_web_sm to enable dependency and entity visualization.")
else:
    examples = [
        "University Metropolitan Tirana has a Faculty of Computer Science and IT.",
        "The university's English admissions page discusses State Matura.",
        "Erasmus+ students may need a Learning Agreement."
    ]
    for sentence in examples:
        doc = trained(sentence)
        print(sentence, [(e.text, e.label_) for e in doc.ents])
        from IPython.display import HTML
        display(HTML(displacy.render(doc, style="dep", jupyter=False)))
        display(HTML(displacy.render(doc, style="ent", jupyter=False)))


University Metropolitan Tirana has a Faculty of Computer Science and IT. [('University Metropolitan Tirana', 'ORG'), ('a Faculty of Computer Science', 'ORG')]


The university's English admissions page discusses State Matura. [('English', 'LANGUAGE'), ('Matura', 'PERSON')]


Erasmus+ students may need a Learning Agreement. [('a Learning Agreement', 'LAW')]


## Submit
1. Three sentences describing the corpus, its source and its genres.
2. Corpus statistics and 10 most frequent words.
3. Three analyzed sentences with one correct and one incorrect NLP annotation each.
4. Source list and copyright/permission notes; explain why outdated rules are flagged.
